In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import os

UPSTREAM_COMMIT = "b76b93da8bd3096a99964a96ae29d46f197a0651"
PROJECT_ROOT = Path("/content/drive/MyDrive/CS2203_MuSc")
RUNS_ROOT = PROJECT_ROOT / "runs"
ACTIVE_RUN_FILE = PROJECT_ROOT / "active-e01-run.txt"

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper",
]

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if ACTIVE_RUN_FILE.is_file():
    candidate_run_id = ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip()
    candidate_root = RUNS_ROOT / candidate_run_id
    if candidate_root.is_dir() and not (candidate_root / "E01_AGGREGATED.json").exists():
        RUN_ID = candidate_run_id
        print("Tiếp tục E01 Run ID đang dở:", RUN_ID)
    else:
        ACTIVE_RUN_FILE.unlink()
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E01_mvtec_all_clip_l14_336_baseline")
        ACTIVE_RUN_FILE.write_text(RUN_ID + "\n", encoding="utf-8")
        print("Tạo E01 Run ID mới:", RUN_ID)
else:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E01_mvtec_all_clip_l14_336_baseline")
    ACTIVE_RUN_FILE.write_text(RUN_ID + "\n", encoding="utf-8")
    print("Tạo E01 Run ID mới:", RUN_ID)

RUN_ROOT = RUNS_ROOT / RUN_ID
CATEGORY_ROOT = RUN_ROOT / "categories"
SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SESSION_ROOT = RUN_ROOT / "sessions" / SESSION_ID

CATEGORY_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_ROOT.mkdir(parents=True, exist_ok=False)

os.environ["MUSC_COMMIT"] = UPSTREAM_COMMIT
os.environ["MUSC_E01_RUN_ROOT"] = str(RUN_ROOT)

(RUN_ROOT / "run-id.txt").write_text(RUN_ID + "\n", encoding="utf-8")

print("Run root:", RUN_ROOT)
print("Session root:", SESSION_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Tiếp tục E01 Run ID đang dở: 20261012_074856_E01_mvtec_all_clip_l14_336_baseline
Run root: /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline
Session root: /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/sessions/20261012_100414


In [ ]:
import platform
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "huggingface_hub",
        "ftfy==6.1.3",
        "openpyxl==3.1.2",
        "timm==0.9.12",
    ],
    check=True,
)

import cv2
import ftfy
import openpyxl
import sklearn
import skimage
import timm
import torch
import torchvision

assert torch.cuda.is_available(), "GPU chưa được bật trong Colab."

versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "opencv": cv2.__version__,
    "ftfy": ftfy.__version__,
    "openpyxl": openpyxl.__version__,
    "scikit_learn": sklearn.__version__,
    "scikit_image": skimage.__version__,
    "timm": timm.__version__,
    "upstream_commit": UPSTREAM_COMMIT,
}

environment_text = "\n".join(f"{key}={value}" for key, value in versions.items())
(SESSION_ROOT / "environment.txt").write_text(environment_text + "\n", encoding="utf-8")
print(environment_text)
subprocess.run(["nvidia-smi"], check=True)

python=3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
platform=Linux-6.6.122+-x86_64-with-glibc2.39
torch=2.11.0+cu130
torchvision=0.26.0+cu130
cuda_runtime=13.0
gpu=Tesla T4
opencv=4.14.0
ftfy=6.1.2
openpyxl=3.1.2
scikit_learn=1.6.1
scikit_image=0.25.2
timm=0.9.12
upstream_commit=b76b93da8bd3096a99964a96ae29d46f197a0651


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
# cell 3 mới nhất
import shutil
import subprocess
from huggingface_hub import HfApi, hf_hub_download

HF_DATASET_REPO = "micguida1/mvtech_anomaly_detection"
HF_DATASET_REVISION = HfApi().dataset_info(HF_DATASET_REPO).sha
HF_CACHE = Path("/content/huggingface-mvtec-cache")

archive_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    filename="mvtec_anomaly_detection.tar.xz",
    revision=HF_DATASET_REVISION,
    cache_dir=HF_CACHE,
)

# Hugging Face dùng symlink trong cache. Cache phải nằm trên filesystem cục bộ,
# không đặt trong Google Drive vì DriveFS có thể biến symlink thành file pointer.
source_archive = Path(archive_path).resolve(strict=True)
local_archive = source_archive
archive_size = source_archive.stat().st_size
content_usage = shutil.disk_usage("/content")
required_free = 8 * 2**30
assert content_usage.free >= required_free, (
    "Không đủ dung lượng cục bộ để giải nén dataset. "
    f"Cần tối thiểu khoảng {required_free / 2**30:.2f} GiB trống."
)
print(f"Archive cục bộ: {archive_size / 2**30:.2f} GiB")

extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

extract_result = subprocess.run(
    ["tar", "-xJf", str(local_archive), "-C", str(extract_root)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("tar_return_code:", extract_result.returncode)
print("tar_output:", extract_result.stdout.strip() or "<empty>")
assert extract_result.returncode == 0, (
    "Giải nén thất bại. Chạy Cell 3D rồi Cell 3R ngay bên dưới."
)

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

for category in CATEGORIES:
    for child in ["train", "test", "ground_truth"]:
        required = DATA_ROOT / category / child
        assert required.is_dir(), f"Thiếu thư mục: {required}"

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
dataset_source_path = RUN_ROOT / "dataset-source.txt"
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=mvtec_anomaly_detection.tar.xz\n"
    f"resolved_root={DATA_ROOT}\n"
)

if dataset_source_path.exists():
    previous_lines = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous_lines[0] == f"source={DATA_SOURCE}", "Dataset revision đã thay đổi khi resume."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Dataset source:", DATA_SOURCE)
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 15 category.")

mvtec_anomaly_detection.tar.xz: reconstructing file:   0%|          |  0.00B / 5.26GB            

mvtec_anomaly_detection.tar.xz: downloading bytes:           |  0.00B            

Archive cục bộ: 4.90 GiB
tar_return_code: 0
tar_output: <empty>
Dataset source: huggingface:micguida1/mvtech_anomaly_detection@eb692b36147a3b3f12c561b879a062104c1f20b8
Dataset root: /content/data
Đã xác nhận đủ 15 category.


In [ ]:
# Khôi phục MVTec AD từ hugging face cache
import shutil
import subprocess
from huggingface_hub import HfApi, hf_hub_download

HF_DATASET_REPO = "micguida1/mvtech_anomaly_detection"
HF_DATASET_REVISION = HfApi().dataset_info(HF_DATASET_REPO).sha
HF_CACHE = PROJECT_ROOT / "cache" / "huggingface"

archive_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    filename="mvtec_anomaly_detection.tar.xz",
    revision=HF_DATASET_REVISION,
    cache_dir=HF_CACHE,
)

extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

subprocess.run(
    ["tar", "-xJf", archive_path, "-C", str(extract_root)],
    check=True,
)

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

for category in CATEGORIES:
    for child in ["train", "test", "ground_truth"]:
        required = DATA_ROOT / category / child
        assert required.is_dir(), f"Thiếu thư mục: {required}"

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
dataset_source_path = RUN_ROOT / "dataset-source.txt"
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=mvtec_anomaly_detection.tar.xz\n"
    f"resolved_root={DATA_ROOT}\n"
)

if dataset_source_path.exists():
    previous_lines = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous_lines[0] == f"source={DATA_SOURCE}", "Dataset revision đã thay đổi khi resume."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Dataset source:", DATA_SOURCE)
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 15 category.")

CalledProcessError: Command '['tar', '-xJf', '/content/drive/MyDrive/CS2203_MuSc/cache/huggingface/datasets--micguida1--mvtech_anomaly_detection/snapshots/eb692b36147a3b3f12c561b879a062104c1f20b8/mvtec_anomaly_detection.tar.xz', '-C', '/content/data']' returned non-zero exit status 2.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

def test_xz(path):
    candidate = Path(path)
    if not candidate.exists() or not candidate.is_file():
        return False, f"File không tồn tại hoặc không phải regular file: {candidate}"
    try:
        with candidate.open("rb") as stream:
            magic = stream.read(6)
    except Exception as error:
        return False, f"Không đọc được archive: {error!r}"
    if magic != bytes.fromhex("fd377a585a00"):
        return False, f"XZ magic không hợp lệ: {magic.hex()}"
    result = subprocess.run(
        ["xz", "--test", str(candidate)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    return result.returncode == 0, result.stdout.strip()

drive_archive = Path(archive_path)
drive_valid, drive_message = test_xz(drive_archive)
print("Drive cache valid:", drive_valid)
print("Drive cache check:", drive_message or "OK")

if not drive_valid:
    print("Cache chưa hợp lệ. Đang tải lại đúng dataset revision...")
    archive_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        repo_type="dataset",
        filename="mvtec_anomaly_detection.tar.xz",
        revision=HF_DATASET_REVISION,
        cache_dir=HF_CACHE,
        force_download=True,
    )
    drive_archive = Path(archive_path)
    drive_valid, drive_message = test_xz(drive_archive)
    print("Drive cache after download:", drive_valid)
    print("Drive cache check after download:", drive_message or "OK")
    assert drive_valid, "Archive vẫn không hợp lệ sau khi tải lại."

archive_size = drive_archive.stat().st_size
content_usage = shutil.disk_usage("/content")
required_free = archive_size + 8 * 2**30
print(f"Archive size: {archive_size / 2**30:.2f} GiB")
print(f"Free space in /content: {content_usage.free / 2**30:.2f} GiB")
assert content_usage.free >= required_free, (
    "Không đủ dung lượng cục bộ để giữ archive và dữ liệu giải nén. "
    f"Cần tối thiểu khoảng {required_free / 2**30:.2f} GiB trống."
)

local_archive = Path("/content/mvtec_anomaly_detection.tar.xz")
if local_archive.exists():
    local_archive.unlink()

print("Đang sao chép archive từ Drive xuống ổ cục bộ...")
shutil.copy2(drive_archive, local_archive)
assert local_archive.stat().st_size == archive_size

local_valid, local_message = test_xz(local_archive)
print("Local archive valid:", local_valid)
print("Local archive check:", local_message or "OK")
assert local_valid, "Bản sao cục bộ không vượt qua kiểm tra XZ."

extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

print("Đang giải nén archive cục bộ...")
extract_result = subprocess.run(
    ["tar", "-xJf", str(local_archive), "-C", str(extract_root)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("tar_return_code:", extract_result.returncode)
print("tar_output:", extract_result.stdout.strip() or "<empty>")
assert extract_result.returncode == 0, "Giải nén archive cục bộ thất bại."

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

for category in CATEGORIES:
    for child in ["train", "test", "ground_truth"]:
        required = DATA_ROOT / category / child
        assert required.is_dir(), f"Thiếu thư mục: {required}"

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
dataset_source_path = RUN_ROOT / "dataset-source.txt"
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=mvtec_anomaly_detection.tar.xz\n"
    f"resolved_root={DATA_ROOT}\n"
    "extraction_source=local_copy_from_drive_cache\n"
)

if dataset_source_path.exists():
    previous_lines = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous_lines[0] == f"source={DATA_SOURCE}", "Dataset revision đã thay đổi."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Phục hồi thành công.")
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 15 category.")

Drive cache valid: False
Drive cache check: XZ magic không hợp lệ: 2e2e2f2e2e2f
Cache chưa hợp lệ. Đang tải lại đúng dataset revision...


mvtec_anomaly_detection.tar.xz: reconstructing file:   0%|          |  0.00B / 5.26GB            

mvtec_anomaly_detection.tar.xz: downloading bytes:           |  0.00B            

Drive cache after download: True
Drive cache check after download: OK
Archive size: 0.00 GiB
Free space in /content: 65.31 GiB
Đang sao chép archive từ Drive xuống ổ cục bộ...
Local archive valid: False
Local archive check: XZ magic không hợp lệ: 2e2e2f2e2e2f


AssertionError: Bản sao cục bộ không vượt qua kiểm tra XZ.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

def test_xz(path):
    candidate = Path(path)
    if not candidate.exists() or not candidate.is_file():
        return False, f"File không tồn tại hoặc không phải regular file: {candidate}"
    try:
        with candidate.open("rb") as stream:
            magic = stream.read(6)
    except Exception as error:
        return False, f"Không đọc được archive: {error!r}"
    if magic != bytes.fromhex("fd377a585a00"):
        return False, f"XZ magic không hợp lệ: {magic.hex()}"
    result = subprocess.run(
        ["xz", "--test", str(candidate)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    return result.returncode == 0, result.stdout.strip()

LOCAL_HF_CACHE = Path("/content/huggingface-mvtec-recovery")
print("Đang tải hoặc dựng lại archive trong filesystem cục bộ của Colab...")
local_download_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    filename="mvtec_anomaly_detection.tar.xz",
    revision=HF_DATASET_REVISION,
    cache_dir=LOCAL_HF_CACHE,
    force_download=True,
)

download_link = Path(local_download_path)
source_archive = download_link.resolve(strict=True)
print("Local download path:", download_link)
print("Resolved archive:", source_archive)

source_valid, source_message = test_xz(source_archive)
print("Local Hugging Face archive valid:", source_valid)
print("Local Hugging Face archive check:", source_message or "OK")
assert source_valid, "Archive cục bộ không vượt qua kiểm tra XZ."

archive_size = source_archive.stat().st_size
content_usage = shutil.disk_usage("/content")
required_free = 8 * 2**30
print(f"Archive size: {archive_size / 2**30:.2f} GiB")
print(f"Free space in /content: {content_usage.free / 2**30:.2f} GiB")
assert content_usage.free >= required_free, (
    "Không đủ dung lượng cục bộ để giải nén dataset. "
    f"Cần tối thiểu khoảng {required_free / 2**30:.2f} GiB trống."
)

extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

print("Đang giải nén archive cục bộ...")
extract_result = subprocess.run(
    ["tar", "-xJf", str(source_archive), "-C", str(extract_root)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("tar_return_code:", extract_result.returncode)
print("tar_output:", extract_result.stdout.strip() or "<empty>")
assert extract_result.returncode == 0, "Giải nén archive cục bộ thất bại."

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

for category in CATEGORIES:
    for child in ["train", "test", "ground_truth"]:
        required = DATA_ROOT / category / child
        assert required.is_dir(), f"Thiếu thư mục: {required}"

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
dataset_source_path = RUN_ROOT / "dataset-source.txt"
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=mvtec_anomaly_detection.tar.xz\n"
    f"resolved_root={DATA_ROOT}\n"
    "extraction_source=colab_local_huggingface_cache\n"
)

if dataset_source_path.exists():
    previous_lines = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous_lines[0] == f"source={DATA_SOURCE}", "Dataset revision đã thay đổi."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Phục hồi thành công.")
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 15 category.")

Đang tải hoặc dựng lại archive trong filesystem cục bộ của Colab...


mvtec_anomaly_detection.tar.xz: reconstructing file:   0%|          |  0.00B / 5.26GB            

mvtec_anomaly_detection.tar.xz: downloading bytes:           |  0.00B            

Local download path: /content/huggingface-mvtec-recovery/datasets--micguida1--mvtech_anomaly_detection/snapshots/eb692b36147a3b3f12c561b879a062104c1f20b8/mvtec_anomaly_detection.tar.xz
Resolved archive: /content/huggingface-mvtec-recovery/blobs/70/7032d0de4ba616af94cccdb31887b653bd2396177ab4cd5fcaebffe491358596
Local Hugging Face archive valid: True
Local Hugging Face archive check: OK
Archive size: 4.90 GiB
Free space in /content: 60.40 GiB
Đang giải nén archive cục bộ...
tar_return_code: 0
tar_output: <empty>
Phục hồi thành công.
Dataset root: /content/data
Đã xác nhận đủ 15 category.


In [ ]:
# Cell 3 mới
import shutil
import subprocess
from huggingface_hub import HfApi, hf_hub_download

HF_DATASET_REPO = "micguida1/mvtech_anomaly_detection"
HF_DATASET_REVISION = HfApi().dataset_info(HF_DATASET_REPO).sha
HF_CACHE = PROJECT_ROOT / "cache" / "huggingface"

archive_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    filename="mvtec_anomaly_detection.tar.xz",
    revision=HF_DATASET_REVISION,
    cache_dir=HF_CACHE,
)

# Google Drive FUSE có thể không ổn định khi tar đọc trực tiếp archive lớn.
# Sao chép archive xuống ổ cục bộ của runtime trước khi giải nén.
drive_archive = Path(archive_path)
local_archive = Path("/content/mvtec_anomaly_detection.tar.xz")
archive_size = drive_archive.stat().st_size
content_usage = shutil.disk_usage("/content")
required_free = archive_size + 8 * 2**30
assert content_usage.free >= required_free, (
    "Không đủ dung lượng cục bộ để giữ archive và dữ liệu giải nén. "
    f"Cần tối thiểu khoảng {required_free / 2**30:.2f} GiB trống."
)

if local_archive.exists():
    local_archive.unlink()
print("Đang sao chép archive từ Drive xuống ổ cục bộ...")
shutil.copy2(drive_archive, local_archive)
assert local_archive.stat().st_size == archive_size

extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

extract_result = subprocess.run(
    ["tar", "-xJf", str(local_archive), "-C", str(extract_root)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("tar_return_code:", extract_result.returncode)
print("tar_output:", extract_result.stdout.strip() or "<empty>")
assert extract_result.returncode == 0, (
    "Giải nén thất bại. Chạy Cell 3D rồi Cell 3R ngay bên dưới."
)

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

for category in CATEGORIES:
    for child in ["train", "test", "ground_truth"]:
        required = DATA_ROOT / category / child
        assert required.is_dir(), f"Thiếu thư mục: {required}"

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
dataset_source_path = RUN_ROOT / "dataset-source.txt"
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=mvtec_anomaly_detection.tar.xz\n"
    f"resolved_root={DATA_ROOT}\n"
)

if dataset_source_path.exists():
    previous_lines = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous_lines[0] == f"source={DATA_SOURCE}", "Dataset revision đã thay đổi khi resume."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Dataset source:", DATA_SOURCE)
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 15 category.")

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

archive = Path(archive_path)

print("archive_path:", archive)
print("exists:", archive.exists())
print("is_file:", archive.is_file())
print("is_symlink:", archive.is_symlink())

if archive.is_symlink():
    print("symlink_target:", os.readlink(archive))

try:
    resolved_archive = archive.resolve(strict=True)
    print("resolved_archive:", resolved_archive)
    print("size_bytes:", resolved_archive.stat().st_size)
    with resolved_archive.open("rb") as stream:
        print("first_6_bytes_hex:", stream.read(6).hex())
except Exception as error:
    print("resolve_or_read_error:", repr(error))

for location in [Path("/content"), Path("/content/drive")]:
    usage = shutil.disk_usage(location)
    print(
        f"disk {location}: "
        f"total={usage.total / 2**30:.2f} GiB, "
        f"used={usage.used / 2**30:.2f} GiB, "
        f"free={usage.free / 2**30:.2f} GiB"
    )

if Path("/content/data").exists():
    partial = subprocess.run(
        ["du", "-sh", "/content/data"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print("partial_extraction:", partial.stdout.strip())

xz_test = subprocess.run(
    ["xz", "--test", str(archive)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("xz_test_return_code:", xz_test.returncode)
print("xz_test_output:", xz_test.stdout.strip() or "<empty>")

archive_path: /content/drive/MyDrive/CS2203_MuSc/cache/huggingface/datasets--micguida1--mvtech_anomaly_detection/snapshots/eb692b36147a3b3f12c561b879a062104c1f20b8/mvtec_anomaly_detection.tar.xz
exists: True
is_file: True
is_symlink: True
symlink_target: ../../blobs/cf4313b13603bec67abb49ca959488f7eedce2a9f7795ec54446c649ac98cd3d
resolved_archive: /content/drive/MyDrive/CS2203_MuSc/cache/huggingface/datasets--micguida1--mvtech_anomaly_detection/blobs/cf4313b13603bec67abb49ca959488f7eedce2a9f7795ec54446c649ac98cd3d
size_bytes: 79
first_6_bytes_hex: 2e2e2f2e2e2f
disk /content: total=112.64 GiB, used=47.31 GiB, free=65.31 GiB
disk /content/drive: total=112.64 GiB, used=50.59 GiB, free=62.04 GiB
partial_extraction: 4.0K	/content/data
xz_test_return_code: 1
xz_test_output: xz: /content/drive/MyDrive/CS2203_MuSc/cache/huggingface/datasets--micguida1--mvtech_anomaly_detection/snapshots/eb692b36147a3b3f12c561b879a062104c1f20b8/mvtec_anomaly_detection.tar.xz: File format not recognized


In [ ]:
# cell 4 clone musc
import shutil
import subprocess

REPO_ROOT = Path("/content/MuSc")
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", "https://github.com/xrli-U/MuSc.git", str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", UPSTREAM_COMMIT],
    cwd=REPO_ROOT,
    check=True,
)

actual_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout.strip()
assert actual_commit == UPSTREAM_COMMIT

entrypoint = REPO_ROOT / "examples" / "musc_main.py"
local_package = REPO_ROOT / "datasets" / "__init__.py"
source = entrypoint.read_bytes()
old_line = b"sys.path.append(os.getcwd())"
new_line = b"sys.path.insert(0, os.getcwd())"

if old_line in source:
    entrypoint.write_bytes(source.replace(old_line, new_line, 1))
elif new_line not in source:
    raise RuntimeError("Không tìm thấy dòng sys.path mong đợi trong musc_main.py")

local_package.touch(exist_ok=True)

diagnostic = subprocess.run(
    [
        sys.executable,
        "-c",
        "import datasets; print(datasets.__file__); "
        "import datasets.mvtec; print('Đã import datasets.mvtec thành công')",
    ],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
print(diagnostic.stdout)

resolved_package = Path(diagnostic.stdout.splitlines()[0]).resolve()
assert REPO_ROOT.resolve() in resolved_package.parents

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
git_diff = subprocess.run(
    ["git", "diff", "--", "examples/musc_main.py"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout

(SESSION_ROOT / "compatibility-fix.txt").write_text(
    "Reason: avoid collision with the Hugging Face datasets package.\n"
    "Effect on method: none; import resolution only.\n\n"
    + git_status
    + "\n"
    + git_diff,
    encoding="utf-8",
)
(RUN_ROOT / "upstream-commit.txt").write_text(actual_commit + "\n", encoding="utf-8")

/content/MuSc/datasets/__init__.py
Đã import datasets.mvtec thành công



41

In [ ]:
# cell 4M - giảm CPU Ram
import subprocess

musc_path = REPO_ROOT / "models" / "musc.py"
source = musc_path.read_text(encoding="utf-8")

def replace_once(text, old, new, label):
    if new in text:
        print(f"SKIP {label}: đã áp dụng")
        return text
    count = text.count(old)
    assert count == 1, f"Không tìm thấy duy nhất marker {label}: count={count}"
    print(f"APPLY {label}")
    return text.replace(old, new, 1)

source = replace_once(
    source,
    "import cv2\n",
    "import cv2\nimport gc\n",
    "import gc",
)

source = replace_once(
    source,
    "        anomaly_maps = anomaly_maps.cpu().numpy()\n"
    "        torch.cuda.empty_cache()\n",
    "        anomaly_maps = anomaly_maps.cpu().numpy()\n"
    "        # Release feature tensors before allocating flattened metric arrays.\n"
    "        del patch_tokens_list, Z_layers, Z, features, patch_tokens\n"
    "        del anomaly_maps_l, anomaly_maps_msm, anomaly_maps_iter\n"
    "        del image_info, image, input_image, test_dataloader, test_dataset, LNAMD_r\n"
    "        gc.collect()\n"
    "        torch.cuda.empty_cache()\n",
    "release expired feature tensors",
)

source = replace_once(
    source,
    "        ac_score = np.array(anomaly_maps).reshape(B, -1).max(-1)\n",
    "        ac_score = anomaly_maps.reshape(B, -1).max(-1)\n",
    "remove anomaly-map copy for image score",
)

source = replace_once(
    source,
    "        pr_px = np.array(anomaly_maps)\n",
    "        pr_px = anomaly_maps\n",
    "remove anomaly-map copy for pixel metrics",
)

musc_path.write_text(source, encoding="utf-8")

memory_diff = subprocess.run(
    ["git", "diff", "--", "models/musc.py"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
assert "gc.collect()" in memory_diff
assert "ac_score = anomaly_maps.reshape" in memory_diff
assert "pr_px = anomaly_maps" in memory_diff

(SESSION_ROOT / "cpu-memory-patch.diff").write_text(memory_diff, encoding="utf-8")
print(memory_diff)

APPLY import gc
APPLY release expired feature tensors
APPLY remove anomaly-map copy for image score
APPLY remove anomaly-map copy for pixel metrics
diff --git a/models/musc.py b/models/musc.py
index 92e91d9..6cd28e3 100644
--- a/models/musc.py
+++ b/models/musc.py
@@ -1,342 +1,348 @@
-import os
-import sys
-import numpy as np
-import torch
-import torch.nn.functional as F
-sys.path.append('./models/backbone')
-
-import datasets.mvtec as mvtec
-from datasets.mvtec import _CLASSNAMES as _CLASSNAMES_mvtec_ad
-import datasets.visa as visa
-from datasets.visa import _CLASSNAMES as _CLASSNAMES_visa
-import datasets.btad as btad
-from datasets.btad import _CLASSNAMES as _CLASSNAMES_btad
-
-import models.backbone.open_clip as open_clip
-import models.backbone._backbones as _backbones
-from models.modules._LNAMD import LNAMD
-from models.modules._MSM import MSM
-from models.modules._RsCIN import RsCIN
-from utils.metrics import compute_metrics
-from openpyxl import Workbook
-from tqdm import tq

In [ ]:
# Cell 5 - Khôi phục CLIP weights cache
import shutil

LOCAL_CLIP_CACHE = Path.home() / ".cache" / "clip"
DRIVE_CLIP_CACHE = PROJECT_ROOT / "cache" / "clip"
LOCAL_CLIP_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CLIP_CACHE.mkdir(parents=True, exist_ok=True)

restored_files = []
for source_file in DRIVE_CLIP_CACHE.glob("*"):
    if source_file.is_file():
        target_file = LOCAL_CLIP_CACHE / source_file.name
        if not target_file.exists() or target_file.stat().st_size != source_file.stat().st_size:
            shutil.copy2(source_file, target_file)
        restored_files.append(target_file.name)

if restored_files:
    print("Đã khôi phục CLIP cache:", restored_files)
else:
    print("Chưa có CLIP cache trong Drive. Category đầu tiên sẽ tải weights khoảng 934 MB.")

Đã khôi phục CLIP cache: ['ViT-L-14-336px.pt']


In [ ]:
# Lưu cấu hình baseline E01
BASELINE_CONFIG = {
    "dataset_name": "mvtec_ad",
    "backbone_name": "ViT-L-14-336",
    "pretrained": "openai",
    "feature_layers": [5, 11, 17, 23],
    "img_resize": 518,
    "divide_num": 1,
    "r_list": [1, 3, 5],
    "batch_size": 4,
    "vis": False,
    "vis_type": "whole_norm",
    "save_excel": True,
    "seed_in_upstream_entrypoint": 42,
}

config_path = RUN_ROOT / "baseline-config.json"
config_text = json.dumps(BASELINE_CONFIG, indent=2, sort_keys=True) + "\n"
if config_path.exists():
    assert config_path.read_text(encoding="utf-8") == config_text, "Baseline config đã thay đổi."
else:
    config_path.write_text(config_text, encoding="utf-8")

print(config_text)

{
  "backbone_name": "ViT-L-14-336",
  "batch_size": 4,
  "dataset_name": "mvtec_ad",
  "divide_num": 1,
  "feature_layers": [
    5,
    11,
    17,
    23
  ],
  "img_resize": 518,
  "pretrained": "openai",
  "r_list": [
    1,
    3,
    5
  ],
  "save_excel": true,
  "seed_in_upstream_entrypoint": 42,
  "vis": false,
  "vis_type": "whole_norm"
}



In [ ]:
# Định nghĩa hàm chạy 1 category
from datetime import datetime, timezone
import json
import pandas as pd
import shlex
import shutil
import subprocess
import time

METRIC_COLUMNS = [
    "auroc_px", "f1_px", "ap_px", "aupro",
    "auroc_sp", "f1_sp", "ap_sp",
]

def sync_clip_cache_to_drive():
    DRIVE_CLIP_CACHE.mkdir(parents=True, exist_ok=True)
    for source_file in LOCAL_CLIP_CACHE.glob("*"):
        if source_file.is_file():
            target_file = DRIVE_CLIP_CACHE / source_file.name
            if not target_file.exists() or target_file.stat().st_size != source_file.stat().st_size:
                shutil.copy2(source_file, target_file)

def successful_attempt(category):
    category_dir = CATEGORY_ROOT / category
    success_files = sorted(category_dir.glob("attempt_*/SUCCESS.json"))
    return success_files[-1].parent if success_files else None

def run_category(category):
    assert category in CATEGORIES, f"Category không hợp lệ: {category}"

    previous_success = successful_attempt(category)
    if previous_success is not None:
        metrics = json.loads((previous_success / "metrics.json").read_text(encoding="utf-8"))
        print(f"SKIP {category}: đã thành công tại {previous_success}")
        return metrics

    category_dir = CATEGORY_ROOT / category
    category_dir.mkdir(parents=True, exist_ok=True)
    attempt_number = len(list(category_dir.glob("attempt_*"))) + 1
    attempt_root = category_dir / f"attempt_{attempt_number:02d}"
    output_root = attempt_root / "output"
    attempt_root.mkdir(parents=True, exist_ok=False)

    command = [
        sys.executable,
        "examples/musc_main.py",
        "--device", "0",
        "--data_path", str(DATA_ROOT),
        "--dataset_name", "mvtec_ad",
        "--class_name", category,
        "--backbone_name", "ViT-L-14-336",
        "--pretrained", "openai",
        "--feature_layers", "5", "11", "17", "23",
        "--img_resize", "518",
        "--divide_num", "1",
        "--r_list", "1", "3", "5",
        "--batch_size", "4",
        "--output_dir", str(output_root),
        "--vis", "False",
        "--vis_type", "whole_norm",
        "--save_excel", "True",
    ]

    command_text = shlex.join(command)
    (attempt_root / "command.sh").write_text(
        "#!/usr/bin/env bash\nset -euo pipefail\n" + command_text + "\n",
        encoding="utf-8",
    )

    started_at = datetime.now(timezone.utc)
    started = time.time()
    log_path = attempt_root / "console.log"

    print(f"START {category}, attempt {attempt_number:02d}")
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()

    elapsed_seconds = time.time() - started
    sync_clip_cache_to_drive()

    status = {
        "category": category,
        "attempt": attempt_number,
        "started_at_utc": started_at.isoformat(),
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": elapsed_seconds,
        "return_code": return_code,
        "gpu": torch.cuda.get_device_name(0),
    }

    if return_code != 0:
        (attempt_root / "FAILED.json").write_text(
            json.dumps(status, indent=2) + "\n",
            encoding="utf-8",
        )
        raise RuntimeError(f"{category} thất bại. Xem log: {log_path}")

    result_file = (
        output_root / "mvtec_ad" / "ViT-L-14-336" /
        "imagesize518" / "results.xlsx"
    )
    assert result_file.is_file(), f"Thiếu file kết quả: {result_file}"

    frame = pd.read_excel(result_file)
    frame = frame.rename(columns={frame.columns[0]: "category"})
    row = frame.iloc[0]
    assert row["category"] == category

    metrics = {"category": category}
    metrics.update({column: float(row[column]) for column in METRIC_COLUMNS})
    metrics["elapsed_seconds"] = elapsed_seconds
    metrics["attempt_root"] = str(attempt_root)

    (attempt_root / "metrics.json").write_text(
        json.dumps(metrics, indent=2) + "\n",
        encoding="utf-8",
    )
    (attempt_root / "SUCCESS.json").write_text(
        json.dumps(status, indent=2) + "\n",
        encoding="utf-8",
    )

    print(f"DONE {category}: {elapsed_seconds / 60:.2f} phút")
    print({column: round(metrics[column], 4) for column in METRIC_COLUMNS})
    return metrics

In [ ]:
# Kiểm tra runtime sạch bằng bottle
E00_BOTTLE = {
    "auroc_px": 98.47773380618383,
    "f1_px": 79.1739992895171,
    "ap_px": 83.04497992850604,
    "aupro": 96.10450992045675,
    "auroc_sp": 99.92063492063492,
    "f1_sp": 99.21259842519686,
    "ap_sp": 99.97519841269839,
}

bottle_metrics = run_category("bottle")
bottle_deltas = {
    metric: bottle_metrics[metric] - E00_BOTTLE[metric]
    for metric in METRIC_COLUMNS
}
max_bottle_delta = max(abs(value) for value in bottle_deltas.values())

print("Chênh lệch với E00:")
for metric, delta in bottle_deltas.items():
    print(f"  {metric}: {delta:+.6f}")

assert max_bottle_delta <= 0.05, (
    f"Bottle lệch quá ngưỡng kiểm tra 0.05 điểm phần trăm: {max_bottle_delta:.6f}"
)

(RUN_ROOT / "clean-session-check.json").write_text(
    json.dumps(
        {
            "category": "bottle",
            "max_absolute_delta_from_E00": max_bottle_delta,
            "deltas": bottle_deltas,
            "status": "passed",
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

print("Kiểm tra runtime sạch đã đạt.")

SKIP bottle: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/categories/bottle/attempt_01
Chênh lệch với E00:
  auroc_px: +0.000000
  f1_px: +0.000000
  ap_px: +0.000000
  aupro: +0.000000
  auroc_sp: +0.000000
  f1_sp: +0.000000
  ap_sp: +0.000000
Kiểm tra runtime sạch đã đạt.


In [ ]:
# cell 8R
from collections import deque

progress_rows = []
for category in CATEGORIES:
    category_dir = CATEGORY_ROOT / category
    attempts = sorted(category_dir.glob("attempt_*"))
    success = successful_attempt(category)

    if success is not None:
        status = "SUCCESS"
        latest = success
    elif attempts:
        latest = attempts[-1]
        status = "FAILED" if (latest / "FAILED.json").is_file() else "INTERRUPTED"
    else:
        latest = None
        status = "NOT_STARTED"

    progress_rows.append(
        {
            "category": category,
            "status": status,
            "attempts": len(attempts),
            "latest_attempt": str(latest) if latest else "",
        }
    )

progress = pd.DataFrame(progress_rows)
display(progress)

interrupted = progress.loc[progress["status"] == "INTERRUPTED", "category"].tolist()
remaining = progress.loc[progress["status"] != "SUCCESS", "category"].tolist()
print("Category bị gián đoạn:", interrupted)
print("Category còn lại:", remaining)

for category in interrupted:
    latest_attempt = Path(
        progress.loc[progress["category"] == category, "latest_attempt"].iloc[0]
    )
    log_path = latest_attempt / "console.log"
    if log_path.is_file():
        with log_path.open("r", encoding="utf-8", errors="replace") as stream:
            tail = list(deque(stream, maxlen=12))
        print(f"\n===== Cuối log của {category} =====")
        print("".join(tail))

,category,status,attempts,latest_attempt
0,bottle,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
1,cable,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
2,capsule,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
3,carpet,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
4,grid,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
5,hazelnut,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
6,leather,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
7,metal_nut,SUCCESS,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
8,pill,INTERRUPTED,1,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
9,screw,NOT_STARTED,0,


Category bị gián đoạn: ['pill']
Category còn lại: ['pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']

===== Cuối log của pill =====
 84%|████████▍ | 141/167 [00:24<00:06,  3.90it/s]



In [ ]:
# Chạy hoặc resume toàn bộ 15 category
all_metrics = {}

for index, category in enumerate(CATEGORIES, start=1):
    print(f"\n===== [{index:02d}/{len(CATEGORIES)}] {category} =====")
    all_metrics[category] = run_category(category)

print("Đã hoàn thành đủ 15 category.")


===== [01/15] bottle =====
SKIP bottle: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/categories/bottle/attempt_01

===== [02/15] cable =====
SKIP cable: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/categories/cable/attempt_01

===== [03/15] capsule =====
SKIP capsule: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/categories/capsule/attempt_01

===== [04/15] carpet =====
SKIP carpet: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/categories/carpet/attempt_01

===== [05/15] grid =====
SKIP grid: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline/categories/grid/attempt_01

===== [06/15] hazelnut =====
SKIP hazelnut: đã thành công tại /content/drive/MyDrive/CS2203_MuSc/runs/202

In [ ]:
import os

required = ["RUN_ID", "RUN_ROOT", "CATEGORY_ROOT", "DATA_ROOT", "REPO_ROOT"]
print({name: name in globals() for name in required})
print("Dataset còn tồn tại:", os.path.isdir(str(globals().get("DATA_ROOT", ""))))

{'RUN_ID': False, 'RUN_ROOT': False, 'CATEGORY_ROOT': False, 'DATA_ROOT': False, 'REPO_ROOT': False}
Dataset còn tồn tại: False


In [ ]:
# Tổng hợp 15 kết quả
records = []
missing_categories = []

for category in CATEGORIES:
    attempt_root = successful_attempt(category)
    if attempt_root is None:
        missing_categories.append(category)
        continue
    metrics = json.loads((attempt_root / "metrics.json").read_text(encoding="utf-8"))
    records.append(metrics)

assert not missing_categories, f"Chưa hoàn thành: {missing_categories}"
assert len(records) == 15

reproduced = pd.DataFrame(records)
reproduced["category"] = pd.Categorical(
    reproduced["category"],
    categories=CATEGORIES,
    ordered=True,
)
reproduced = reproduced.sort_values("category").reset_index(drop=True)

mean_row = {"category": "mean"}
for column in METRIC_COLUMNS:
    mean_row[column] = reproduced[column].mean()
mean_row["elapsed_seconds"] = reproduced["elapsed_seconds"].sum()
mean_row["attempt_root"] = ""

reproduced_with_mean = pd.concat(
    [reproduced, pd.DataFrame([mean_row])],
    ignore_index=True,
)

reproduced_csv = RUN_ROOT / "mvtec_ad_reproduced.csv"
reproduced_xlsx = RUN_ROOT / "mvtec_ad_reproduced.xlsx"
reproduced_with_mean.to_csv(reproduced_csv, index=False)
reproduced_with_mean.to_excel(reproduced_xlsx, index=False)

display(reproduced_with_mean)
print("Tổng thời gian category subprocess:", reproduced["elapsed_seconds"].sum() / 60, "phút")

,category,auroc_px,f1_px,ap_px,aupro,auroc_sp,f1_sp,ap_sp,elapsed_seconds,attempt_root
0,bottle,98.477734,79.173999,83.044980,96.104510,99.920635,99.212598,99.975198,343.690234,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
1,cable,95.760713,60.965664,57.694474,89.615690,98.931784,97.297297,99.386437,701.534113,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
2,capsule,98.956415,49.800676,48.500718,95.563777,96.689270,95.693780,99.347707,559.324703,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
3,carpet,99.453789,73.330835,76.052416,97.581886,99.879615,99.441341,99.962123,465.178630,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
4,grid,98.161632,43.933751,38.236802,93.913731,98.663325,96.491228,99.537192,248.140128,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
5,hazelnut,99.375670,73.409799,73.288920,92.230007,99.607143,98.550725,99.790693,411.327857,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
6,leather,99.717012,62.845035,64.472530,98.734479,100.000000,100.000000,100.000000,500.398922,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
7,metal_nut,86.119333,46.220935,47.539157,89.343504,96.578690,97.382199,99.146209,483.314481,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
8,pill,97.473372,65.531816,67.247384,98.007988,96.126568,95.890411,99.288722,820.972347,/content/drive/MyDrive/CS2203_MuSc/runs/202610...
9,screw,98.767382,41.877120,36.113518,94.401294,82.311949,88.888889,90.951184,743.429550,/content/drive/MyDrive/CS2203_MuSc/runs/202610...


Tổng thời gian category subprocess: 119.49076574643453 phút


In [ ]:
#Cell 11 - So sánh với repository chính thức
reference_records = [
    {"category": "bottle", "auroc_sp": 99.92, "f1_sp": 99.21, "ap_sp": 99.98, "auroc_px": 98.48, "f1_px": 79.17, "ap_px": 83.04, "aupro": 96.10},
    {"category": "cable", "auroc_sp": 98.99, "f1_sp": 97.30, "ap_sp": 99.42, "auroc_px": 95.76, "f1_px": 60.97, "ap_px": 57.70, "aupro": 89.62},
    {"category": "capsule", "auroc_sp": 96.45, "f1_sp": 94.88, "ap_sp": 99.30, "auroc_px": 98.96, "f1_px": 49.80, "ap_px": 48.45, "aupro": 95.49},
    {"category": "carpet", "auroc_sp": 99.88, "f1_sp": 99.44, "ap_sp": 99.96, "auroc_px": 99.45, "f1_px": 73.33, "ap_px": 76.05, "aupro": 97.58},
    {"category": "grid", "auroc_sp": 98.66, "f1_sp": 96.49, "ap_sp": 99.54, "auroc_px": 98.16, "f1_px": 43.94, "ap_px": 38.24, "aupro": 93.92},
    {"category": "hazelnut", "auroc_sp": 99.61, "f1_sp": 98.55, "ap_sp": 99.79, "auroc_px": 99.38, "f1_px": 73.41, "ap_px": 73.28, "aupro": 92.24},
    {"category": "leather", "auroc_sp": 100.00, "f1_sp": 100.00, "ap_sp": 100.00, "auroc_px": 99.72, "f1_px": 62.84, "ap_px": 64.47, "aupro": 98.74},
    {"category": "metal_nut", "auroc_sp": 96.92, "f1_sp": 97.38, "ap_sp": 99.25, "auroc_px": 86.12, "f1_px": 46.22, "ap_px": 47.54, "aupro": 89.34},
    {"category": "pill", "auroc_sp": 96.24, "f1_sp": 95.89, "ap_sp": 99.31, "auroc_px": 97.47, "f1_px": 65.54, "ap_px": 67.25, "aupro": 98.01},
    {"category": "screw", "auroc_sp": 82.17, "f1_sp": 88.89, "ap_sp": 90.88, "auroc_px": 98.77, "f1_px": 41.87, "ap_px": 36.12, "aupro": 94.40},
    {"category": "tile", "auroc_sp": 100.00, "f1_sp": 100.00, "ap_sp": 100.00, "auroc_px": 97.90, "f1_px": 74.71, "ap_px": 78.90, "aupro": 94.64},
    {"category": "toothbrush", "auroc_sp": 100.00, "f1_sp": 100.00, "ap_sp": 100.00, "auroc_px": 99.53, "f1_px": 70.19, "ap_px": 67.79, "aupro": 95.48},
    {"category": "transistor", "auroc_sp": 99.42, "f1_sp": 95.00, "ap_sp": 99.19, "auroc_px": 91.38, "f1_px": 59.24, "ap_px": 58.40, "aupro": 77.21},
    {"category": "wood", "auroc_sp": 98.51, "f1_sp": 98.33, "ap_sp": 99.52, "auroc_px": 97.24, "f1_px": 68.64, "ap_px": 74.75, "aupro": 94.50},
    {"category": "zipper", "auroc_sp": 99.84, "f1_sp": 99.17, "ap_sp": 99.96, "auroc_px": 98.40, "f1_px": 62.48, "ap_px": 61.89, "aupro": 94.46},
]

repository_mean = {
    "category": "mean",
    "auroc_sp": 97.77,
    "f1_sp": 97.37,
    "ap_sp": 99.07,
    "auroc_px": 97.11,
    "f1_px": 62.16,
    "ap_px": 62.26,
    "aupro": 93.45,
}

reference = pd.DataFrame(reference_records + [repository_mean])
comparison = reproduced_with_mean[["category", *METRIC_COLUMNS]].merge(
    reference[["category", *METRIC_COLUMNS]],
    on="category",
    suffixes=("_reproduced", "_reference"),
    validate="one_to_one",
)

for metric in METRIC_COLUMNS:
    comparison[f"{metric}_delta"] = (
        comparison[f"{metric}_reproduced"] - comparison[f"{metric}_reference"]
    )

delta_columns = [f"{metric}_delta" for metric in METRIC_COLUMNS]
category_rows = comparison[comparison["category"] != "mean"]
max_absolute_delta = category_rows[delta_columns].abs().to_numpy().max()

alerts = []
for _, row in category_rows.iterrows():
    for metric in METRIC_COLUMNS:
        delta = float(row[f"{metric}_delta"])
        if abs(delta) > 0.10:
            alerts.append({
                "category": row["category"],
                "metric": metric,
                "reproduced": row[f"{metric}_reproduced"],
                "reference": row[f"{metric}_reference"],
                "delta": delta,
            })

comparison.to_csv(RUN_ROOT / "mvtec_ad_reference_comparison.csv", index=False)
comparison.to_excel(RUN_ROOT / "mvtec_ad_reference_comparison.xlsx", index=False)
pd.DataFrame(alerts).to_csv(RUN_ROOT / "comparison-alerts.csv", index=False)

mean_comparison = comparison[comparison["category"] == "mean"]
display(mean_comparison)
print(f"Sai khác tuyệt đối lớn nhất ở category rows: {max_absolute_delta:.6f}")
print(f"Số cảnh báo vượt 0.10 điểm phần trăm: {len(alerts)}")

summary_lines = [
    "# E01 MVTec AD Baseline Summary",
    "",
    f"- Run ID: `{RUN_ID}`",
    f"- Completed categories: {len(records)}/15",
    f"- Maximum absolute category-metric delta: {max_absolute_delta:.6f} percentage points",
    f"- Deltas above 0.10 percentage points: {len(alerts)}",
    "",
    "## Reproduced Mean",
    "",
]

mean_record = reproduced_with_mean[reproduced_with_mean["category"] == "mean"].iloc[0]
for metric in METRIC_COLUMNS:
    summary_lines.append(f"- {metric}: {mean_record[metric]:.6f}")

(RUN_ROOT / "E01_SUMMARY.md").write_text(
    "\n".join(summary_lines) + "\n",
    encoding="utf-8",
)

completion = {
    "run_id": RUN_ID,
    "aggregated_at_utc": datetime.now(timezone.utc).isoformat(),
    "completed_categories": len(records),
    "max_absolute_category_metric_delta": float(max_absolute_delta),
    "alerts_over_0_10_percentage_points": len(alerts),
}
(RUN_ROOT / "E01_AGGREGATED.json").write_text(
    json.dumps(completion, indent=2) + "\n",
    encoding="utf-8",
)

if ACTIVE_RUN_FILE.is_file() and ACTIVE_RUN_FILE.read_text(encoding="utf-8").strip() == RUN_ID:
    ACTIVE_RUN_FILE.unlink()

,category,auroc_px_reproduced,f1_px_reproduced,ap_px_reproduced,aupro_reproduced,auroc_sp_reproduced,f1_sp_reproduced,ap_sp_reproduced,auroc_px_reference,f1_px_reference,...,auroc_sp_reference,f1_sp_reference,ap_sp_reference,auroc_px_delta,f1_px_delta,ap_px_delta,aupro_delta,auroc_sp_delta,f1_sp_delta,ap_sp_delta
15,mean,97.11439,62.157971,62.261741,93.45296,97.734031,97.405687,99.052802,97.11,62.16,...,97.77,97.37,99.07,0.00439,-0.002029,0.001741,0.00296,-0.035969,0.035687,-0.017198


Sai khác tuyệt đối lớn nhất ở category rows: 0.813780
Số cảnh báo vượt 0.10 điểm phần trăm: 10


In [ ]:
# Đóng gói kết quả
import shutil

EXPORT_ROOT = PROJECT_ROOT / "exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
archive_base = EXPORT_ROOT / RUN_ID
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=RUN_ROOT,
)

print("Đã tạo ZIP:", archive_path)

Đã tạo ZIP: /content/drive/MyDrive/CS2203_MuSc/exports/20261012_074856_E01_mvtec_all_clip_l14_336_baseline.zip


In [ ]:
## E01Q - Qualitative Analysis
#cell q0
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from datetime import datetime, timezone
from pathlib import Path
import json
import os

UPSTREAM_COMMIT = "b76b93da8bd3096a99964a96ae29d46f197a0651"
PROJECT_ROOT = Path("/content/drive/MyDrive/CS2203_MuSc")
RUNS_ROOT = PROJECT_ROOT / "runs"
EXPORT_ROOT = PROJECT_ROOT / "exports"
SOURCE_E01_RUN_ID = "20261012_074856_E01_mvtec_all_clip_l14_336_baseline"
SOURCE_E01_ROOT = RUNS_ROOT / SOURCE_E01_RUN_ID
ACTIVE_Q_RUN_FILE = PROJECT_ROOT / "active-e01q-run.txt"

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper",
]
QUAL_CATEGORIES = ["leather", "transistor", "screw"]

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

if ACTIVE_Q_RUN_FILE.is_file():
    candidate_id = ACTIVE_Q_RUN_FILE.read_text(encoding="utf-8").strip()
    candidate_root = RUNS_ROOT / candidate_id
    if candidate_root.is_dir() and not (candidate_root / "E01Q_DONE.json").exists():
        RUN_ID = candidate_id
        print("Tiếp tục E01Q Run ID:", RUN_ID)
    else:
        ACTIVE_Q_RUN_FILE.unlink()
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E01Q_mvtec_qualitative")
        ACTIVE_Q_RUN_FILE.write_text(RUN_ID + "\n", encoding="utf-8")
        print("Tạo E01Q Run ID mới:", RUN_ID)
else:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_E01Q_mvtec_qualitative")
    ACTIVE_Q_RUN_FILE.write_text(RUN_ID + "\n", encoding="utf-8")
    print("Tạo E01Q Run ID mới:", RUN_ID)

RUN_ROOT = RUNS_ROOT / RUN_ID
CATEGORY_ROOT = RUN_ROOT / "categories"
SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SESSION_ROOT = RUN_ROOT / "sessions" / SESSION_ID

CATEGORY_ROOT.mkdir(parents=True, exist_ok=True)
SESSION_ROOT.mkdir(parents=True, exist_ok=False)

assert SOURCE_E01_ROOT.is_dir(), f"Không tìm thấy E01 nguồn: {SOURCE_E01_ROOT}"
assert (SOURCE_E01_ROOT / "E01_AGGREGATED.json").is_file()

print("E01Q run root:", RUN_ROOT)
print("E01 source root:", SOURCE_E01_ROOT)

Mounted at /content/drive
Tạo E01Q Run ID mới: 20261012_151018_E01Q_mvtec_qualitative
E01Q run root: /content/drive/MyDrive/CS2203_MuSc/runs/20261012_151018_E01Q_mvtec_qualitative
E01 source root: /content/drive/MyDrive/CS2203_MuSc/runs/20261012_074856_E01_mvtec_all_clip_l14_336_baseline


In [ ]:
# cell q1
from pathlib import Path

if "DATA_ROOT" not in globals():
    DATA_ROOT = Path("/content/data/mvtec_anomaly_detection")
    if not DATA_ROOT.is_dir():
        DATA_ROOT = Path("/content/data")

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path("/content/MuSc")

dataset_ready = (
    DATA_ROOT.is_dir()
    and all((DATA_ROOT / category / "test").is_dir() for category in QUAL_CATEGORIES)
)
repo_ready = (REPO_ROOT / "examples" / "musc_main.py").is_file()
memory_patch_ready = (
    repo_ready
    and "gc.collect()" in (REPO_ROOT / "models" / "musc.py").read_text(
        encoding="utf-8", errors="replace"
    )
)

print("Dataset ready:", dataset_ready, DATA_ROOT)
print("Repository ready:", repo_ready, REPO_ROOT)
print("CPU memory patch ready:", memory_patch_ready)

if not (dataset_ready and repo_ready and memory_patch_ready):
    raise RuntimeError(
        "Runtime E01 không còn đầy đủ. Thực hiện mục 5 bên dưới, "
        "sau đó chạy lại Cell Q1."
    )

Dataset ready: True /content/data
Repository ready: True /content/MuSc
CPU memory patch ready: True


In [ ]:
#e01 cell 2
import platform
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "huggingface_hub",
        "ftfy==6.1.3",
        "openpyxl==3.1.2",
        "timm==0.9.12",
    ],
    check=True,
)

import cv2
import ftfy
import openpyxl
import sklearn
import skimage
import timm
import torch
import torchvision

assert torch.cuda.is_available(), "GPU chưa được bật trong Colab."

versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "opencv": cv2.__version__,
    "ftfy": ftfy.__version__,
    "openpyxl": openpyxl.__version__,
    "scikit_learn": sklearn.__version__,
    "scikit_image": skimage.__version__,
    "timm": timm.__version__,
    "upstream_commit": UPSTREAM_COMMIT,
}

environment_text = "\n".join(f"{key}={value}" for key, value in versions.items())
(SESSION_ROOT / "environment.txt").write_text(environment_text + "\n", encoding="utf-8")
print(environment_text)
subprocess.run(["nvidia-smi"], check=True)

python=3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
platform=Linux-6.6.122+-x86_64-with-glibc2.39
torch=2.11.0+cu130
torchvision=0.26.0+cu130
cuda_runtime=13.0
gpu=Tesla T4
opencv=4.14.0
ftfy=6.1.2
openpyxl=3.1.2
scikit_learn=1.6.1
scikit_image=0.25.2
timm=0.9.12
upstream_commit=b76b93da8bd3096a99964a96ae29d46f197a0651


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
#cell 3 bản mới
import shutil
import subprocess
from huggingface_hub import HfApi, hf_hub_download

HF_DATASET_REPO = "micguida1/mvtech_anomaly_detection"
HF_DATASET_REVISION = HfApi().dataset_info(HF_DATASET_REPO).sha
HF_CACHE = Path("/content/huggingface-mvtec-cache")

archive_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    filename="mvtec_anomaly_detection.tar.xz",
    revision=HF_DATASET_REVISION,
    cache_dir=HF_CACHE,
)

# Hugging Face dùng symlink trong cache. Cache phải nằm trên filesystem cục bộ,
# không đặt trong Google Drive vì DriveFS có thể biến symlink thành file pointer.
source_archive = Path(archive_path).resolve(strict=True)
local_archive = source_archive
archive_size = source_archive.stat().st_size
content_usage = shutil.disk_usage("/content")
required_free = 8 * 2**30
assert content_usage.free >= required_free, (
    "Không đủ dung lượng cục bộ để giải nén dataset. "
    f"Cần tối thiểu khoảng {required_free / 2**30:.2f} GiB trống."
)
print(f"Archive cục bộ: {archive_size / 2**30:.2f} GiB")

extract_root = Path("/content/data")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True, exist_ok=False)

extract_result = subprocess.run(
    ["tar", "-xJf", str(local_archive), "-C", str(extract_root)],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print("tar_return_code:", extract_result.returncode)
print("tar_output:", extract_result.stdout.strip() or "<empty>")
assert extract_result.returncode == 0, (
    "Giải nén thất bại. Chạy Cell 3D rồi Cell 3R ngay bên dưới."
)

expected_root = extract_root / "mvtec_anomaly_detection"
if expected_root.is_dir():
    DATA_ROOT = expected_root
else:
    candidates = [path.parent.parent for path in extract_root.rglob("bottle/test")]
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, f"Không xác định được duy nhất dataset root: {candidates}"
    DATA_ROOT = candidates[0]

for category in CATEGORIES:
    for child in ["train", "test", "ground_truth"]:
        required = DATA_ROOT / category / child
        assert required.is_dir(), f"Thiếu thư mục: {required}"

DATA_SOURCE = f"huggingface:{HF_DATASET_REPO}@{HF_DATASET_REVISION}"
dataset_source_path = RUN_ROOT / "dataset-source.txt"
dataset_source_text = (
    f"source={DATA_SOURCE}\n"
    "archive=mvtec_anomaly_detection.tar.xz\n"
    f"resolved_root={DATA_ROOT}\n"
)

if dataset_source_path.exists():
    previous_lines = dataset_source_path.read_text(encoding="utf-8").splitlines()
    assert previous_lines[0] == f"source={DATA_SOURCE}", "Dataset revision đã thay đổi khi resume."
else:
    dataset_source_path.write_text(dataset_source_text, encoding="utf-8")

print("Dataset source:", DATA_SOURCE)
print("Dataset root:", DATA_ROOT)
print("Đã xác nhận đủ 15 category.")

mvtec_anomaly_detection.tar.xz: reconstructing file:   0%|          |  0.00B / 5.26GB            

mvtec_anomaly_detection.tar.xz: downloading bytes:           |  0.00B            

Archive cục bộ: 4.90 GiB
tar_return_code: 0
tar_output: <empty>
Dataset source: huggingface:micguida1/mvtech_anomaly_detection@eb692b36147a3b3f12c561b879a062104c1f20b8
Dataset root: /content/data
Đã xác nhận đủ 15 category.


In [ ]:
# cell 4
import shutil
import subprocess

REPO_ROOT = Path("/content/MuSc")
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ["git", "clone", "https://github.com/xrli-U/MuSc.git", str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", UPSTREAM_COMMIT],
    cwd=REPO_ROOT,
    check=True,
)

actual_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout.strip()
assert actual_commit == UPSTREAM_COMMIT

entrypoint = REPO_ROOT / "examples" / "musc_main.py"
local_package = REPO_ROOT / "datasets" / "__init__.py"
source = entrypoint.read_bytes()
old_line = b"sys.path.append(os.getcwd())"
new_line = b"sys.path.insert(0, os.getcwd())"

if old_line in source:
    entrypoint.write_bytes(source.replace(old_line, new_line, 1))
elif new_line not in source:
    raise RuntimeError("Không tìm thấy dòng sys.path mong đợi trong musc_main.py")

local_package.touch(exist_ok=True)

diagnostic = subprocess.run(
    [
        sys.executable,
        "-c",
        "import datasets; print(datasets.__file__); "
        "import datasets.mvtec; print('Đã import datasets.mvtec thành công')",
    ],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
print(diagnostic.stdout)

resolved_package = Path(diagnostic.stdout.splitlines()[0]).resolve()
assert REPO_ROOT.resolve() in resolved_package.parents

git_status = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
git_diff = subprocess.run(
    ["git", "diff", "--", "examples/musc_main.py"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout

(SESSION_ROOT / "compatibility-fix.txt").write_text(
    "Reason: avoid collision with the Hugging Face datasets package.\n"
    "Effect on method: none; import resolution only.\n\n"
    + git_status
    + "\n"
    + git_diff,
    encoding="utf-8",
)
(RUN_ROOT / "upstream-commit.txt").write_text(actual_commit + "\n", encoding="utf-8")

/content/MuSc/datasets/__init__.py
Đã import datasets.mvtec thành công



41

In [ ]:
#cell 4M
import subprocess

musc_path = REPO_ROOT / "models" / "musc.py"
source = musc_path.read_bytes()
line_ending = b"\r\n" if b"\r\n" in source else b"\n"

def encode_block(text):
    return text.encode("utf-8").replace(b"\n", line_ending)

def replace_once(data, old_text, new_text, label):
    old = encode_block(old_text)
    new = encode_block(new_text)
    if new in data:
        print(f"SKIP {label}: đã áp dụng")
        return data
    count = data.count(old)
    assert count == 1, f"Không tìm thấy duy nhất marker {label}: count={count}"
    print(f"APPLY {label}")
    return data.replace(old, new, 1)

source = replace_once(
    source,
    "import cv2\n",
    "import cv2\nimport gc\n",
    "import gc",
)

source = replace_once(
    source,
    "        anomaly_maps = anomaly_maps.cpu().numpy()\n"
    "        torch.cuda.empty_cache()\n",
    "        anomaly_maps = anomaly_maps.cpu().numpy()\n"
    "        # Release feature tensors before allocating flattened metric arrays.\n"
    "        del patch_tokens_list, Z_layers, Z, features, patch_tokens\n"
    "        del anomaly_maps_l, anomaly_maps_msm, anomaly_maps_iter\n"
    "        del image_info, image, input_image, test_dataloader, test_dataset, LNAMD_r\n"
    "        gc.collect()\n"
    "        torch.cuda.empty_cache()\n",
    "release expired feature tensors",
)

source = replace_once(
    source,
    "        ac_score = np.array(anomaly_maps).reshape(B, -1).max(-1)\n",
    "        ac_score = anomaly_maps.reshape(B, -1).max(-1)\n",
    "remove anomaly-map copy for image score",
)

source = replace_once(
    source,
    "        pr_px = np.array(anomaly_maps)\n",
    "        pr_px = anomaly_maps\n",
    "remove anomaly-map copy for pixel metrics",
)

musc_path.write_bytes(source)

memory_diff = subprocess.run(
    ["git", "diff", "--", "models/musc.py"],
    cwd=REPO_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    check=True,
).stdout
assert "gc.collect()" in memory_diff
assert "ac_score = anomaly_maps.reshape" in memory_diff
assert "pr_px = anomaly_maps" in memory_diff

(SESSION_ROOT / "cpu-memory-patch.diff").write_text(memory_diff, encoding="utf-8")
print(memory_diff)

APPLY import gc
APPLY release expired feature tensors
APPLY remove anomaly-map copy for image score
APPLY remove anomaly-map copy for pixel metrics
diff --git a/models/musc.py b/models/musc.py
index 92e91d9..a94213f 100644
--- a/models/musc.py
+++ b/models/musc.py
@@ -23,6 +23,7 @@ from tqdm import tqdm
 import pickle
 import time
 import cv2
+import gc
 
 import warnings
 warnings.filterwarnings("ignore")
@@ -234,10 +235,15 @@ class MuSc():
         print('MuSc: {}ms per image'.format((end_time_all-start_time_all)*1000/dataset_num))
 
         anomaly_maps = anomaly_maps.cpu().numpy()
+        # Release feature tensors before allocating flattened metric arrays.
+        del patch_tokens_list, Z_layers, Z, features, patch_tokens
+        del anomaly_maps_l, anomaly_maps_msm, anomaly_maps_iter
+        del image_info, image, input_image, test_dataloader, test_dataset, LNAMD_r
+        gc.collect()
         torch.cuda.empty_cache()
 
         B = anomaly_maps.shape[0]   # the number of u

In [ ]:
# cell 5
import shutil

LOCAL_CLIP_CACHE = Path.home() / ".cache" / "clip"
DRIVE_CLIP_CACHE = PROJECT_ROOT / "cache" / "clip"
LOCAL_CLIP_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CLIP_CACHE.mkdir(parents=True, exist_ok=True)

restored_files = []
for source_file in DRIVE_CLIP_CACHE.glob("*"):
    if source_file.is_file():
        target_file = LOCAL_CLIP_CACHE / source_file.name
        if not target_file.exists() or target_file.stat().st_size != source_file.stat().st_size:
            shutil.copy2(source_file, target_file)
        restored_files.append(target_file.name)

if restored_files:
    print("Đã khôi phục CLIP cache:", restored_files)
else:
    print("Chưa có CLIP cache trong Drive. Category đầu tiên sẽ tải weights khoảng 934 MB.")

Đã khôi phục CLIP cache: ['ViT-L-14-336px.pt']


In [ ]:
# cell q2
from datetime import datetime, timezone
import json
import pandas as pd
import shlex
import subprocess
import sys
import time

METRIC_COLUMNS = [
    "auroc_px", "f1_px", "ap_px", "aupro",
    "auroc_sp", "f1_sp", "ap_sp",
]

QUAL_CONFIG = {
    "source_e01_run_id": SOURCE_E01_RUN_ID,
    "categories": QUAL_CATEGORIES,
    "dataset_name": "mvtec_ad",
    "backbone_name": "ViT-L-14-336",
    "pretrained": "openai",
    "feature_layers": [5, 11, 17, 23],
    "img_resize": 518,
    "divide_num": 1,
    "r_list": [1, 3, 5],
    "batch_size": 4,
    "vis": True,
    "vis_type": "whole_norm",
    "save_excel": True,
    "seed_in_upstream_entrypoint": 42,
}

(RUN_ROOT / "qualitative-config.json").write_text(
    json.dumps(QUAL_CONFIG, indent=2) + "\n",
    encoding="utf-8",
)

source_metrics = pd.read_csv(SOURCE_E01_ROOT / "mvtec_ad_reproduced.csv")

def successful_qual_attempt(category):
    success_files = sorted((CATEGORY_ROOT / category).glob("attempt_*/SUCCESS.json"))
    return success_files[-1].parent if success_files else None

def run_qual_category(category):
    previous = successful_qual_attempt(category)
    if previous is not None:
        metrics = json.loads((previous / "metrics.json").read_text(encoding="utf-8"))
        print(f"SKIP {category}: đã thành công tại {previous}")
        return metrics

    category_dir = CATEGORY_ROOT / category
    category_dir.mkdir(parents=True, exist_ok=True)
    attempt_number = len(list(category_dir.glob("attempt_*"))) + 1
    attempt_root = category_dir / f"attempt_{attempt_number:02d}"
    output_root = attempt_root / "output"
    attempt_root.mkdir(parents=True, exist_ok=False)

    command = [
        sys.executable,
        "examples/musc_main.py",
        "--device", "0",
        "--data_path", str(DATA_ROOT),
        "--dataset_name", "mvtec_ad",
        "--class_name", category,
        "--backbone_name", "ViT-L-14-336",
        "--pretrained", "openai",
        "--feature_layers", "5", "11", "17", "23",
        "--img_resize", "518",
        "--divide_num", "1",
        "--r_list", "1", "3", "5",
        "--batch_size", "4",
        "--output_dir", str(output_root),
        "--vis", "True",
        "--vis_type", "whole_norm",
        "--save_excel", "True",
    ]

    (attempt_root / "command.sh").write_text(
        "#!/usr/bin/env bash\nset -euo pipefail\n" + shlex.join(command) + "\n",
        encoding="utf-8",
    )

    started_at = datetime.now(timezone.utc)
    started = time.time()
    log_path = attempt_root / "console.log"
    print(f"START {category}, attempt {attempt_number:02d}")

    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_ROOT,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()

    elapsed_seconds = time.time() - started
    status = {
        "category": category,
        "attempt": attempt_number,
        "started_at_utc": started_at.isoformat(),
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_seconds": elapsed_seconds,
        "return_code": return_code,
    }

    if return_code != 0:
        (attempt_root / "FAILED.json").write_text(
            json.dumps(status, indent=2) + "\n", encoding="utf-8"
        )
        raise RuntimeError(f"{category} thất bại. Xem {log_path}")

    result_root = (
        output_root / "mvtec_ad" / "ViT-L-14-336" / "imagesize518"
    )
    result_file = result_root / "results.xlsx"
    heatmap_root = result_root / category
    assert result_file.is_file(), f"Thiếu {result_file}"
    assert heatmap_root.is_dir(), f"Thiếu {heatmap_root}"

    heatmaps = [
        path for path in heatmap_root.rglob("*")
        if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"}
    ]
    expected_images = list((DATA_ROOT / category / "test").glob("*/*"))
    assert len(heatmaps) == len(expected_images), (
        f"Số heatmap không khớp: {len(heatmaps)} != {len(expected_images)}"
    )

    frame = pd.read_excel(result_file)
    frame = frame.rename(columns={frame.columns[0]: "category"})
    row = frame.iloc[0]
    assert row["category"] == category

    metrics = {"category": category}
    metrics.update({column: float(row[column]) for column in METRIC_COLUMNS})
    metrics["elapsed_seconds"] = elapsed_seconds
    metrics["heatmap_count"] = len(heatmaps)
    metrics["attempt_root"] = str(attempt_root)

    source_row = source_metrics[source_metrics["category"] == category].iloc[0]
    deltas = {
        column: metrics[column] - float(source_row[column])
        for column in METRIC_COLUMNS
    }
    max_delta = max(abs(value) for value in deltas.values())
    metrics["max_absolute_delta_from_e01"] = max_delta
    assert max_delta <= 0.05, (
        f"{category} lệch E01 quá 0.05 điểm phần trăm: {max_delta}"
    )

    (attempt_root / "metrics.json").write_text(
        json.dumps(metrics, indent=2) + "\n", encoding="utf-8"
    )
    (attempt_root / "SUCCESS.json").write_text(
        json.dumps(status, indent=2) + "\n", encoding="utf-8"
    )
    print(f"DONE {category}: {elapsed_seconds / 60:.2f} phút, {len(heatmaps)} heatmap")
    print("Max delta from E01:", max_delta)
    return metrics

In [ ]:
#cell q3
qual_metrics = {}
for index, category in enumerate(QUAL_CATEGORIES, start=1):
    print(f"\n===== [{index}/{len(QUAL_CATEGORIES)}] {category} =====")
    qual_metrics[category] = run_qual_category(category)

print("Đã hoàn thành inference định tính cho ba category.")


===== [1/3] leather =====
START leather, attempt 01
xFormers not available
xFormers not available
{'datasets': {'dataset_name': 'mvtec_ad', 'data_path': '/content/data', 'class_name': 'leather', 'img_resize': 518, 'divide_num': 1}, 'models': {'backbone_name': 'ViT-L-14-336', 'pretrained': 'openai', 'batch_size': 4, 'feature_layers': [5, 11, 17, 23], 'r_list': [1, 3, 5]}, 'device': '0', 'testing': {'output_dir': '/content/drive/MyDrive/CS2203_MuSc/runs/20261012_151018_E01Q_mvtec_qualitative/categories/leather/attempt_01/output', 'vis': True, 'vis_type': 'whole_norm', 'save_excel': True}}
leather

100%|██████████| 31/31 [00:31<00:00,  1.00s/it]
extract time: 250.5775555487602ms per image
aggregation degree: 1
LNAMD-1: 26.873430898112634ms per image
layer-0 mutual scoring...

100%|██████████| 124/124 [00:12<00:00,  9.76it/s]
layer-1 mutual scoring...

100%|██████████| 124/124 [00:13<00:00,  9.46it/s]
layer-2 mutual scoring...

100%|██████████| 124/124 [00:13<00:00,  9.09it/s]
layer-3 mut

In [ ]:
# cell q4 - Khôi phục giá trị heatmap và xếp hạng case
import cv2
import numpy as np
from sklearn.metrics import roc_auc_score

jet_lut = cv2.applyColorMap(
    np.arange(256, dtype=np.uint8).reshape(256, 1),
    cv2.COLORMAP_JET,
).reshape(256, 3)

lut_keys = (
    jet_lut[:, 0].astype(np.uint32)
    | (jet_lut[:, 1].astype(np.uint32) << 8)
    | (jet_lut[:, 2].astype(np.uint32) << 16)
)
inverse_lut = np.full(1 << 24, -1, dtype=np.int16)
inverse_lut[lut_keys] = np.arange(256, dtype=np.int16)

def heatmap_to_score(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    assert bgr is not None, f"Không đọc được heatmap: {path}"
    keys = (
        bgr[:, :, 0].astype(np.uint32)
        | (bgr[:, :, 1].astype(np.uint32) << 8)
        | (bgr[:, :, 2].astype(np.uint32) << 16)
    )
    score = inverse_lut[keys]
    assert np.all(score >= 0), f"Heatmap chứa màu ngoài JET LUT: {path}"
    return score.astype(np.float32) / 255.0

records = []
for category in QUAL_CATEGORIES:
    attempt_root = successful_qual_attempt(category)
    assert attempt_root is not None
    heatmap_root = (
        attempt_root / "output" / "mvtec_ad" / "ViT-L-14-336" /
        "imagesize518" / category
    )

    heatmaps = sorted(
        path for path in heatmap_root.rglob("*")
        if path.is_file() and path.suffix.lower() in {".png", ".jpg", ".jpeg"}
    )
    for heatmap_path in heatmaps:
        anomaly_type = heatmap_path.parent.name
        image_name = heatmap_path.name
        image_path = DATA_ROOT / category / "test" / anomaly_type / image_name
        assert image_path.is_file(), f"Thiếu ảnh gốc: {image_path}"

        score = heatmap_to_score(heatmap_path)
        flat = score.reshape(-1)
        top_count = max(1, int(np.ceil(flat.size * 0.01)))
        top1_mean = float(np.partition(flat, -top_count)[-top_count:].mean())

        mask_path = ""
        pixel_auroc = np.nan
        if anomaly_type != "good":
            mask_candidates = sorted(
                (DATA_ROOT / category / "ground_truth" / anomaly_type).glob(
                    f"{Path(image_name).stem}*"
                )
            )
            assert len(mask_candidates) == 1, (
                f"Không tìm thấy duy nhất mask cho {image_path}: {mask_candidates}"
            )
            mask_path = str(mask_candidates[0])
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (score.shape[1], score.shape[0]), interpolation=cv2.INTER_NEAREST)
            target = (mask > 0).astype(np.uint8).reshape(-1)
            assert target.min() == 0 and target.max() == 1
            pixel_auroc = float(roc_auc_score(target, flat))

        records.append(
            {
                "category": category,
                "anomaly_type": anomaly_type,
                "image_name": image_name,
                "image_path": str(image_path),
                "mask_path": mask_path,
                "heatmap_path": str(heatmap_path),
                "pixel_auroc_from_saved_heatmap": pixel_auroc,
                "top1_percent_heatmap_mean": top1_mean,
            }
        )

case_scores = pd.DataFrame(records)
case_scores.to_csv(RUN_ROOT / "qualitative-case-scores.csv", index=False)

selected_parts = []
for category in QUAL_CATEGORIES:
    category_rows = case_scores[case_scores["category"] == category]
    anomaly_rows = category_rows[category_rows["anomaly_type"] != "good"]
    normal_rows = category_rows[category_rows["anomaly_type"] == "good"]

    best = anomaly_rows.nlargest(2, "pixel_auroc_from_saved_heatmap").copy()
    best["selection_reason"] = "best_localization"
    worst = anomaly_rows.nsmallest(2, "pixel_auroc_from_saved_heatmap").copy()
    worst["selection_reason"] = "worst_localization"
    normal_fp = normal_rows.nlargest(2, "top1_percent_heatmap_mean").copy()
    normal_fp["selection_reason"] = "high_response_normal_candidate"

    selected_parts.extend([best, worst, normal_fp])

selected = pd.concat(selected_parts, ignore_index=True)
selected = selected.drop_duplicates(
    subset=["category", "anomaly_type", "image_name"], keep="first"
)
selected.to_csv(RUN_ROOT / "selected-qualitative-cases.csv", index=False)

display(
    selected[
        [
            "category", "anomaly_type", "image_name", "selection_reason",
            "pixel_auroc_from_saved_heatmap", "top1_percent_heatmap_mean",
        ]
    ]
)

,category,anomaly_type,image_name,selection_reason,pixel_auroc_from_saved_heatmap,top1_percent_heatmap_mean
0,leather,glue,011.png,best_localization,0.999944,0.629071
1,leather,poke,010.png,best_localization,0.999940,0.642500
2,leather,poke,004.png,worst_localization,0.987857,0.739264
3,leather,color,015.png,worst_localization,0.992038,0.819573
4,leather,good,013.png,high_response_normal_candidate,NaN,0.415701
5,leather,good,031.png,high_response_normal_candidate,NaN,0.404160
6,transistor,damaged_case,002.png,best_localization,0.999752,0.632596
7,transistor,bent_lead,004.png,best_localization,0.999681,0.642347
8,transistor,cut_lead,008.png,worst_localization,0.595726,0.575049
9,transistor,cut_lead,001.png,worst_localization,0.643028,0.540851


In [ ]:
# cell q5- Tạo asset và panel cho báo cáo
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

ASSET_ROOT = RUN_ROOT / "selected-assets"
ASSET_ROOT.mkdir(parents=True, exist_ok=True)

def prepare_original(path):
    image = Image.open(path).convert("RGB")
    image = TF.resize(
        image,
        518,
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    )
    image = TF.center_crop(image, [518, 518])
    return np.asarray(image)

asset_records = []
for _, row in selected.iterrows():
    key = f"{row['category']}__{row['anomaly_type']}__{Path(row['image_name']).stem}"
    case_root = ASSET_ROOT / key
    case_root.mkdir(parents=True, exist_ok=True)

    original = prepare_original(row["image_path"])
    heatmap_bgr = cv2.imread(row["heatmap_path"], cv2.IMREAD_COLOR)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB)
    if heatmap_rgb.shape[:2] != original.shape[:2]:
        heatmap_rgb = cv2.resize(heatmap_rgb, (original.shape[1], original.shape[0]))

    if row["mask_path"]:
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (original.shape[1], original.shape[0]), interpolation=cv2.INTER_NEAREST)
    else:
        mask = np.zeros(original.shape[:2], dtype=np.uint8)

    overlay = cv2.addWeighted(original, 0.55, heatmap_rgb, 0.45, 0)
    contours, _ = cv2.findContours(
        (mask > 0).astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )
    cv2.drawContours(overlay, contours, -1, (255, 255, 255), 2)

    Image.fromarray(original).save(case_root / "original.png")
    Image.fromarray(mask).save(case_root / "ground_truth.png")
    Image.fromarray(heatmap_rgb).save(case_root / "heatmap.png")
    Image.fromarray(overlay).save(case_root / "overlay.png")

    fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title("Ground truth")
    axes[2].imshow(heatmap_rgb)
    axes[2].set_title("MuSc heatmap")
    axes[3].imshow(overlay)
    axes[3].set_title("Overlay + GT contour")
    for axis in axes:
        axis.axis("off")

    px_auc = row["pixel_auroc_from_saved_heatmap"]
    px_text = "N/A" if pd.isna(px_auc) else f"{px_auc:.4f}"
    fig.suptitle(
        f"{row['category']} | {row['anomaly_type']} | {row['image_name']} | "
        f"{row['selection_reason']} | saved-map pixel AUROC={px_text}",
        fontsize=11,
    )
    fig.tight_layout()
    panel_path = case_root / "panel.png"
    fig.savefig(panel_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

    asset_records.append(
        {
            **row.to_dict(),
            "asset_root": str(case_root),
            "panel_path": str(panel_path),
        }
    )

asset_index = pd.DataFrame(asset_records)
asset_index.to_csv(RUN_ROOT / "selected-assets-index.csv", index=False)
print("Đã tạo", len(asset_index), "panel tại", ASSET_ROOT)

Đã tạo 18 panel tại /content/drive/MyDrive/CS2203_MuSc/runs/20261012_151018_E01Q_mvtec_qualitative/selected-assets


In [ ]:
# cell q6 - xem nhanh các panel
from IPython.display import display

for category in QUAL_CATEGORIES:
    print(f"\n===== {category} =====")
    rows = asset_index[asset_index["category"] == category]
    for _, row in rows.iterrows():
        print(
            row["selection_reason"],
            row["anomaly_type"],
            row["image_name"],
        )
        display(Image.open(row["panel_path"]))

In [ ]:
# cell q7 - hoàn tất và tạo zip nhỏ
import zipfile

done = {
    "run_id": RUN_ID,
    "source_e01_run_id": SOURCE_E01_RUN_ID,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "categories": QUAL_CATEGORIES,
    "selected_case_count": int(len(asset_index)),
    "status": "completed",
}
(RUN_ROOT / "E01Q_DONE.json").write_text(
    json.dumps(done, indent=2) + "\n", encoding="utf-8"
)

zip_path = EXPORT_ROOT / f"{RUN_ID}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_ROOT.rglob("*"):
        if not path.is_file():
            continue
        relative = path.relative_to(RUN_ROOT)
        if "output" in relative.parts:
            continue
        archive.write(path, arcname=str(relative))

assert zip_path.is_file()
print("E01Q hoàn tất.")
print("ZIP:", zip_path)
print("Dung lượng ZIP:", zip_path.stat().st_size / 2**20, "MiB")

E01Q hoàn tất.
ZIP: /content/drive/MyDrive/CS2203_MuSc/exports/20261012_151018_E01Q_mvtec_qualitative.zip
Dung lượng ZIP: 36.263726234436035 MiB
